In [77]:
import xml.etree.ElementTree as ET
import numpy as np
file = open('./drstickler.svg')
tree = ET.parse(file)
root = tree.getroot()
file.close()
print('root:', root)

root: <Element '{http://www.w3.org/2000/svg}svg' at 0x00000271E446F100>


In [78]:

paths = root.findall('.//{http://www.w3.org/2000/svg}path')
def line_segments_from_paths(paths):
    # Pairwise list of line endpoints
    lines = []
    ''' Expects path data to be in the form of a list of points, with no curves. 
    Either
    d="m 143.86517,100.11236 -0.74158,46.34831 -13.16292,10.9382 -5.19101,12.79214 -7.60112,-4.26405"
    or 
    d="m 143.86517,100.11236 -0.74158,46.34831 -13.16292,10.9382 -5.19101,12.79214 -7.60112,-4.26405 z"
    '''
    for path in paths:
        id = path.attrib['id']
        # print('parsing path',id)
        points = path.attrib['d'].split(' ')[1:]
        # print ('points:', points)
        
        # Only used if the path ends with 'z', which indicates a closed path. In that case, we want to add the first point to the end of the list of points.
        firstpoint = points[0]

        # this will be a list of lists of length 2, where each inner list is a point [x,y]
        pointlist = []

        y_min = 1000000
        y_max = -1000000
        for point in points:
            xyz = []
            if point == 'h' or point == 'H' or point == 'v' or point == 'V':
                print(id,'contains a curve command, which is not supported. Skipping path.')
                break
            if point == 'z' or point == 'Z':
                # print(id,'ends with z = closed path.')
                xyz = firstpoint.split(',') + [0]
                pointlist.append(xyz)
            else:
                xyz = point.split(',') + [0]
                pointlist.append(xyz)
                y = float(point.split(',')[1])
                if y < y_min:
                    y_min = y
                if y > y_max:
                    y_max = y

        
        # print('pointlist:', pointlist)

        for i in range(len(pointlist)-1):
            lines.append([pointlist[i], pointlist[i+1]])

    # print('lines:', lines)
    lines = np.array(lines).astype(float)
    return lines, y_min, y_max

def normalize_lines(lines, y_min, y_max):
    # Normalize the lines to be between 0 and 1 in the y direction, and between 0 and 1 in the x direction. 
    # We can do this by first translating the lines so that the minimum y value is at 0, and then scaling the lines so that the maximum y value is at 1. 
    # We can also scale the x values by the same factor as the y values, so that the aspect ratio of the image is preserved.
    lines[:,:,1] = (lines[:,:,1] - y_min) / (y_max - y_min)
    lines[:,:,0] = lines[:,:,0] / (y_max - y_min)
    return lines
lines, y_min, y_max = line_segments_from_paths(paths)
print('lines shape:', lines.shape)
print('y_min:', y_min)
print('y_max:', y_max)
    

lines shape: (22, 2, 3)
y_min: -15.57304
y_max: 137.5618


array([[145.53371 ,  60.808988],
       [167.22472 ,  73.41573 ]])